# Recreating the temperature sensitivity analysis of Evan & Eisenman (2021)

Recreation of [Evan, A. & Eisenman, I. (2021), *A mechanism for regional variations in snowpack melt under rising temperature*, Nature Climate Change 11, 326-330](https://doi.org/10.1038/s41558-021-00996-w), for comparison against this repo's spring temperature sensitivity analysis (`mountain_range_era5_analysis.ipynb`, `sierra_nevada.ipynb`).

**Evan & Eisenman (2021), hereafter E&E21, in one paragraph.** The snowpack disappearance date $\zeta$ (first water-year day with SWE $=0$ after the seasonal peak) shifts earlier as temperature rises, but *not uniformly*: SNOTEL observations show $-30$ days $^\circ$C$^{-1}$ in some regions and near zero in others. The authors explain this with an idealized model in which daily temperature follows a sinusoid (their Eq. 1),

$$T(t) = T_0 - T_1 \sin(\omega t - \phi),$$

snow accumulates at rate $R_a$ when $T<T_m$ and melts at rate $R_m$ when $T>T_m$. Solving for $\zeta$ and differentiating gives their Eq. 2:

$$\frac{\partial \zeta}{\partial T_0} = -\frac{1}{\omega}\frac{1}{\sqrt{T_1^2 - (T_0 - T_m)^2}}\left(1 + \frac{R_a}{R_m}\right),$$

i.e. the sensitivity is controlled by how close the annual-mean temperature $T_0$ is to the amplitude of the annual cycle $T_1$ (relative to the melt threshold $T_m$). Where $T$ crosses $T_m$ near the flat crest/trough of the sinusoid (warm coasts, cold coasts: $T_1 \approx |T_0 - T_m|$), a small warming removes many below-freezing days $\rightarrow$ large sensitivity. Where the crossing is on the steep part of the sinusoid (continental interiors: $T_1 \gg |T_0 - T_m|$), sensitivity is small. Applied globally, this predicts the fastest changes along coasts, the Arctic, the western US, Central Europe, and southern South America.

Original MATLAB code: [amatoevan/snowpack_zeta](https://github.com/amatoevan/snowpack_zeta).

## What is recreated, and with which data

| E&E21 element | Original source | This notebook |
|---|---|---|
| E&E21 Fig. 1 — two water years at Virginia Lakes Ridge | NRCS SNOTEL daily SWE | identical data via the [egagli/snotel_ccss_stations](https://github.com/egagli/snotel_ccss_stations) archive (reproduces E&E21's numbers exactly) |
| E&E21 Fig. 2 — observed vs idealized $\partial\zeta/\partial T_0$, 398 stations, NARR $T$,$P$, WY 1982-2018 | SNOTEL + NARR | SNOTEL-measured $T$ and $P$, WY 2001-2018 — E&E21's own measured-data variant (their Supplementary Fig. 2a: $r=0.64$, 363 stations); NARR interpolation is not recreated |
| E&E21 Fig. 3a — station-by-station obs vs Eq. 2 | as above | recreated |
| E&E21 Fig. 3b — VIC hydrologic model check | VIC simulations | **not recreated** (requires running the VIC land-surface model) |
| E&E21 Fig. 4 — global $\partial\zeta/\partial T_0$ | MERRA-2 monthly climatology 1982-2018, 0.5$^\circ$ | ERA5 daily climatology 1990-2019 at $\sim$0.7$^\circ$ ([WeatherBench2](https://weatherbench2.readthedocs.io/en/latest/data-guide.html), anonymous HTTPS — no credentials needed) |
| E&E21 Extended Data Fig. 1 / Supp. Fig. 4 — idealized-model intuition | analytic | recreated |

**First-run downloads** (all cached locally afterwards, no credentials required): SNOTEL archive $\sim$200 MB compressed ($\sim$460 MB of CSVs in `data/snotel_ccss_archive/`), ERA5 climatology $\sim$770 MB streamed once (reduced to a $\sim$2 MB netCDF in `era5_data/`).

In [ ]:
import glob
import pathlib
import subprocess

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio.features
import rasterio.transform
import xarray as xr

from gsro_analysis import paths, settings, stats as gsro_stats
from cartopy.io import shapereader
from scipy import stats
from tqdm.auto import tqdm

import easysnowdata

OMEGA = 2 * np.pi / 365.0  # annual angular frequency [rad/day]

# station analysis parameters (E&E21 values noted inline)
SWE_ZERO_CM = 0.1                        # "snow gone" threshold [cm] (E&E21 Fig1.m)
PEAK_MIN_CM, PEAK_MAX_CM = 10.0, 500.0   # valid peak-SWE range [cm] (E&E21 Fig2.m)
WY_START, WY_END = 2001, 2018            # E&E21's measured-T/P variant period (Supp. Fig. 2a)
MIN_JOINT_YEARS = 17                     # near-continuous records, like E&E21's station selection
MAX_MISSING_DAYS = 30                    # E&E21: "missing for fewer than 30 days" per water year

# global map parameters (E&E21 Fig4.m)
RAM_GLOBAL = 0.34   # station-averaged Ra/Rm
TM_GLOBAL = 0.18    # station-averaged Tm [degC]

SNOTEL_DIR = paths.DATA / 'snotel_ccss_archive'
ERA5_CLIM_NC = paths.ERA5 / 'evan_eisenman_2021_era5_T0_T1_climatology.nc'
config = settings.load_config()  # dataset version from settings.CONFIG_FILE
STATION_METRICS_CSV = paths.resultsdir('climate/temperature_sensitivity_comparison', config.version) / 'evan_eisenman_2021_snotel_station_metrics.csv'
FIG_DIR = paths.figdir('climate/temperature_sensitivity_comparison', config.version)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. The idealized model

In the model, snow accumulates at a constant rate $R_a$ on days with $T<T_m$ (while precipitation lasts, $t<t_p$) and melts at a constant rate $R_m$ on days with $T>T_m$. Setting total accumulation equal to total melt and solving for the disappearance date $\zeta$ gives

$$\zeta = \left[\frac{\pi+\phi}{\omega} + \frac{R_a}{R_m}\left(t_p - \frac{\phi}{\omega}\right)\right] - \frac{1}{\omega}\sin^{-1}\!\left(\frac{T_0-T_m}{T_1}\right)\left(1+\frac{R_a}{R_m}\right),$$

whose derivative with respect to $T_0$ is Eq. 2 above. Two properties drive everything that follows:

* $\partial\zeta/\partial T_0$ is **symmetric about $T_0 - T_m = 0$** — warm maritime and cold Arctic climates can be equally sensitive;
* it **diverges as $|T_0 - T_m| \rightarrow T_1$** — the annual cycle grazes the melt threshold near its flat crest or trough, so a small warming removes many freezing days.

In [ ]:
def sine_temperature(t, T0, T1, phi=0.0):
    """Eq. 1: idealized annual temperature cycle. t in water-year days (t=0 on 1 Oct)."""
    return T0 - T1 * np.sin(OMEGA * t - phi)


def dzeta_dT0(T0, T1, ram=RAM_GLOBAL, tm=TM_GLOBAL):
    """Eq. 2: sensitivity of the snow disappearance date to annual-mean temperature
    [days per degC]. NaN where the annual cycle does not cross Tm (T1 <= |T0 - Tm|)."""
    T0, T1 = np.asarray(T0, dtype=float), np.asarray(T1, dtype=float)
    with np.errstate(invalid='ignore'):
        out = -(1.0 / OMEGA) * (1.0 + ram) / np.sqrt(T1**2 - (T0 - tm) ** 2)
    return np.where(np.abs(T0 - tm) < T1, out, np.nan)

### E&E21 Extended Data Fig. 1 — why the shape of the annual cycle matters

Three climate regimes with identical warming (+1 $^\circ$C), following E&E21's schematic (here with $T_m=0$, $R_a/R_m=0.34$). The shaded below-freezing season shrinks by tens of days per degree where the sinusoid grazes 0 $^\circ$C (warm/cold coastal), but barely changes where the crossing is steep (continental interior).

In [ ]:
regimes = [
    ('Warm coastal (e.g. Pacific Northwest)', 4.0, 6.0),
    ('Continental interior (e.g. Colorado Rockies)', 0.0, 18.0),
    ('Cold coastal (e.g. coastal Arctic)', -4.0, 6.0),
]
t = np.arange(0, 366)
tm = 0.0

fig, axes = plt.subplots(3, 2, figsize=(10, 8), sharex=True)
for row, (name, T0, T1) in zip(axes, regimes):
    for ax, dT0 in zip(row, [0.0, 1.0]):
        T = sine_temperature(t, T0 + dT0, T1)
        days_below = (np.pi - 2 * np.arcsin((T0 + dT0 - tm) / T1)) / OMEGA
        ax.plot(t, T, color='k', lw=1.5)
        ax.axhline(tm, color='0.5', lw=0.8)
        ax.fill_between(t, T, tm, where=T < tm, color='tab:blue', alpha=0.3)
        ax.fill_between(t, T, tm, where=T >= tm, color='tab:red', alpha=0.15)
        ax.set_title(f'{name}\n$T_0$={T0 + dT0:+.0f}, $T_1$={T1:.0f} | {days_below:.0f} days below $T_m$'
                     if dT0 == 0 else rf'+1$^\circ$C warming | {days_below:.0f} days below $T_m$',
                     fontsize=9)
        ax.set_xticks([0, 92, 182, 273, 365])
        ax.set_xticklabels(['Oct 1', 'Jan 1', 'Apr 1', 'Jul 1', 'Oct 1'], fontsize=8)
    sens = dzeta_dT0(T0, T1, tm=tm)
    row[1].annotate(rf'$\partial\zeta/\partial T_0$ = {sens:.0f} d $^\circ$C$^{{-1}}$',
                    xy=(1.03, 0.5), xycoords='axes fraction', fontsize=10, va='center')
for ax in axes[:, 0]:
    ax.set_ylabel(r'T ($^\circ$C)')
fig.suptitle('Recreation of Evan & Eisenman (2021) Extended Data Fig. 1', y=1.005)
fig.tight_layout()
fig.savefig(FIG_DIR / 'ee2021_annual_cycle_schematic.png', dpi=250, bbox_inches='tight')
plt.show()

### Parameter space of Eq. 2 (cf. E&E21's Supplementary Figs. 4-5)

Sensitivity as a function of $T_0 - T_m$ (how far the annual mean sits from the melt threshold) and $T_1$ (annual cycle amplitude). White = undefined (temperature never crosses the threshold). The sensitivity is symmetric about $T_0-T_m=0$ and diverges toward the $T_1 = |T_0 - T_m|$ boundary.

In [ ]:
t0_grid = np.linspace(-20, 20, 401)
t1_grid = np.linspace(0.25, 25, 200)
G0, G1 = np.meshgrid(t0_grid, t1_grid)
sens_grid = dzeta_dT0(G0 + TM_GLOBAL, G1)

fig, ax = plt.subplots(figsize=(8, 5))
pc = ax.pcolormesh(t0_grid, t1_grid, sens_grid, vmin=-40, vmax=0, cmap='viridis', shading='auto')
cs = ax.contour(t0_grid, t1_grid, sens_grid, levels=[-30, -20, -10, -5], colors='w', linewidths=0.8)
ax.clabel(cs, fmt='%.0f', fontsize=8)
ax.plot(t0_grid, np.abs(t0_grid), color='k', lw=1, ls='--', label='$T_1 = |T_0 - T_m|$ (undefined below)')
ax.set_xlabel(r'$T_0 - T_m$ ($^\circ$C)')
ax.set_ylabel(r'$T_1$ ($^\circ$C)')
ax.set_ylim(0, 25)
ax.legend(loc='upper center')
fig.colorbar(pc, label=r'$\partial\zeta/\partial T_0$ (days $^\circ$C$^{-1}$)')
ax.set_title('Eq. 2 sensitivity in parameter space ($R_a/R_m$ = 0.34)')
fig.savefig(FIG_DIR / 'ee2021_eq2_parameter_space.png', dpi=250, bbox_inches='tight')
plt.show()

## 2. SNOTEL station data

E&E21 used daily SNOTEL SWE (via [PANGAEA 896396](https://doi.pangaea.de/10.1594/PANGAEA.896396), the dataset of Evan 2019). Here we pull the same NRCS measurements from the [egagli/snotel_ccss_stations](https://github.com/egagli/snotel_ccss_stations) archive (updated daily, used throughout this project via `easysnowdata`): daily `WTEQ` (SWE, m), `TAVG` ($^\circ$C), and `PRCPSA` (daily precipitation increment, m). The full-archive download happens once and is cached in `data/snotel_ccss_archive/`.

In [ ]:
station_collection = easysnowdata.automatic_weather_stations.StationCollection()
all_stations = station_collection.all_stations
# contiguous western US, as in E&E21 (their Fig1.m keeps lat < 50)
west = all_stations[(all_stations.network == 'SNOTEL')
                    & (all_stations.geometry.y < 50)
                    & (all_stations.geometry.y > 31)]

csv_dir = SNOTEL_DIR / 'data'
if not csv_dir.exists() or not list(csv_dir.glob('*.csv')):
    SNOTEL_DIR.mkdir(parents=True, exist_ok=True)
    archive = SNOTEL_DIR / 'all_station_data.tar.lzma'
    if not archive.exists():
        print('Downloading SNOTEL/CCSS archive (~200 MB)...')
        subprocess.run(['wget', '-q', '-O', str(archive),
                        'https://github.com/egagli/snotel_ccss_stations/raw/main/data/all_station_data.tar.lzma'],
                       check=True)
    print('Extracting...')
    subprocess.run(['tar', '--lzma', '-xf', str(archive), '-C', str(SNOTEL_DIR)], check=True)

station_csvs = {pathlib.Path(f).stem: f for f in glob.glob(str(csv_dir / '*.csv'))}
west = west[west.index.isin(station_csvs)]
print(f'{len(west)} western US (31-50N) SNOTEL stations with data '
      f'(E&E21: 398 long-record stations; 363 in the measured-T/P variant)')

In [ ]:
def wy_daily(df, wy):
    """365-day water-year slice (WY day 1 = 1 Oct; day 366 dropped in leap years, as in E&E21)."""
    idx = pd.date_range(f'{wy - 1}-10-01', f'{wy}-09-30')[:365]
    return df.reindex(idx)


def snow_season_metrics(swe_cm):
    """Peak SWE p [cm] and the 1-based water-year days of snow appearance f,
    peak m, and disappearance zeta, following E&E21's Fig1.m/Fig2.m:
    m = mean day of the maximum, f/zeta = last/first day with SWE < 0.1 cm
    before/after the peak. Returns None if the year fails QC."""
    s = swe_cm.values if hasattr(swe_cm, 'values') else swe_cm
    if np.all(np.isnan(s)) or np.isnan(s).sum() >= MAX_MISSING_DAYS:
        return None
    p = np.nanmax(s)
    if not (PEAK_MIN_CM < p < PEAK_MAX_CM):  # E&E21 Fig2.m: reject p<10 or p>500 cm
        return None
    m = int(round(np.where(s == p)[0].mean()))
    before = np.where((s < SWE_ZERO_CM) & (np.arange(365) < m))[0]
    after = np.where((s < SWE_ZERO_CM) & (np.arange(365) > m))[0]
    if len(before) == 0 or len(after) == 0:
        return None
    f, z = before[-1], after[0]
    # E&E21 Fig2.m QC: accumulation and melt seasons must not be >=90% missing-or-zero
    for lo, hi in [(f, m), (m, z)]:
        seg = s[lo + 1:hi]
        if len(seg) == 0 or np.isnan(seg).sum() >= 0.9 * len(seg) or (seg == 0).sum() >= 0.9 * len(seg):
            return None
    return p, f + 1, m + 1, z + 1


def fit_sine(doy, temps):
    """Least-squares fit of Eq. 1 via T(d) = c0 + c1 sin(wd) + c2 cos(wd).
    Returns (T0, T1, rmse); E&E21's fitsine() is the same model."""
    ok = np.isfinite(temps)
    X = np.column_stack([np.ones(ok.sum()), np.sin(OMEGA * doy[ok]), np.cos(OMEGA * doy[ok])])
    coef, *_ = np.linalg.lstsq(X, temps[ok], rcond=None)
    rmse = np.sqrt(np.mean((X @ coef - temps[ok]) ** 2))
    return coef[0], np.hypot(coef[1], coef[2]), rmse


def smooth11(x):
    """11-day running mean, as applied to SWE in E&E21 (Supp. Fig. 1g-h)."""
    return pd.Series(x).rolling(11, center=True, min_periods=6).mean().values

## 3. E&E21 Fig. 1 — two contrasting water years at Virginia Lakes Ridge

SNOTEL 846 (38$^\circ$N 119$^\circ$W, Sierra Nevada): WY 2014 (dry, warm) vs WY 2017 (wet, cool). $R_a$ = peak SWE / accumulation duration, $R_m$ = peak SWE / melt duration. Since this is the same underlying NRCS data E&E21 used, the recreated $\zeta$, $R_a$ and $R_m$ should match the published values exactly.

In [ ]:
vlr = pd.read_csv(station_csvs['846_CA_SNTL'], index_col=0, parse_dates=True)
paper_fig1 = {2017: dict(zeta=260, Ra=0.69, Rm=1.80, color='tab:blue'),
              2014: dict(zeta=226, Ra=0.17, Rm=0.56, color='tab:red')}

fig, ax = plt.subplots(figsize=(9, 6))
t = np.arange(1, 366)
for wy, ref in paper_fig1.items():
    s = (wy_daily(vlr, wy)['WTEQ'] * 100).values  # m -> cm
    p, f, m, z = snow_season_metrics(s)
    Ra, Rm = p / (m - f), p / (z - m)
    c = ref['color']
    ax.plot(t, s, color=c, lw=2, label=str(wy))
    ax.plot([f, m], [0, p], ls='--', color=c, lw=1.2, label=f'$R_a$={Ra:.2f} cm d$^{{-1}}$')
    ax.plot([m, z], [p, 0], ls='-.', color=c, lw=1.2, label=f'$R_m$={Rm:.2f} cm d$^{{-1}}$')
    ax.annotate(f'$\\zeta_{{{wy}}}$={z}', xy=(z, 0), xytext=(z - 75, p * 0.3 + 8),
                fontsize=11, arrowprops=dict(arrowstyle='->', lw=0.8))
    print(f'WY{wy}: peak={p:5.0f} cm  zeta={z} (E&E21 {ref["zeta"]})  '
          f'Ra={Ra:.2f} (E&E21 {ref["Ra"]:.2f})  Rm={Rm:.2f} (E&E21 {ref["Rm"]:.2f})')

ax.set_xlim(30, 270); ax.set_xticks(np.arange(30, 271, 30))
ax.set_ylim(0, 120); ax.set_yticks(np.arange(0, 121, 15))
ax.grid(alpha=0.3)
ax.set_xlabel('$t$ (water-year day)')
ax.set_ylabel('$S$ (cm)')
ax.set_title('Virginia Lakes Ridge SNOTEL station (38$^\\circ$N 119$^\\circ$W)\n'
             'Recreation of Evan & Eisenman (2021) Fig. 1')
ax.legend(loc='upper left', fontsize=9)
fig.savefig(FIG_DIR / 'ee2021_virginia_lakes_ridge_two_water_years.png', dpi=250, bbox_inches='tight')
plt.show()

## 4. Per-station metrics: observed and idealized sensitivity

For every western SNOTEL station and water year we compute $\zeta$, peak SWE, $R_a$, $R_m$, water-year mean $T$ and total $P$; per station we then compute

* **observed sensitivity** $\partial\zeta/\partial T_0$: multilinear regression $\zeta \sim [1,\ T_0,\ P_0]$ across years (E&E21's Eq. on p. 326), with the 95% CI on the temperature coefficient;
* **$T_0$, $T_1$**: sinusoid fit to the long-term-mean daily $T$ cycle;
* **$R_a/R_m$**: mean over years of the annual ratio;
* **$T_m$**: coolest long-term-mean daily temperature on melting days (declining long-term-mean SWE). Deviation from E&E21: we use the 11-day-smoothed SWE cycle and only days after the climatological peak — with an 18-year (rather than 37-year) record the raw long-term mean is noisy enough that ephemeral autumn melt events otherwise produce spuriously cold $T_m$;
* **idealized sensitivity** (Eq. 2) from $T_0$, $T_1$, station-mean $T_m$, and per-station $R_a/R_m$, exactly as in E&E21's Fig2.m.

Like E&E21's measured-data variant we use SNOTEL-measured $T$/$P$ for WY 2001-2018 and require near-continuous records ($\geq$ 17 of 18 years). Delete `csvs/evan_eisenman_2021_snotel_station_metrics.csv` to recompute from scratch.

In [ ]:
if STATION_METRICS_CSV.exists():
    station_df = pd.read_csv(STATION_METRICS_CSV, index_col=0)
    print(f'Loaded cached metrics for {len(station_df)} stations from {STATION_METRICS_CSV}')
else:
    doy = np.arange(1, 366, dtype=float)
    records = []
    for code_ in tqdm(west.index, desc='stations'):
        df = pd.read_csv(station_csvs[code_], index_col=0, parse_dates=True)
        if not {'WTEQ', 'TAVG', 'PRCPSA'}.issubset(df.columns):
            continue
        zeta, ra, rm, twy, pwy = [], [], [], [], []
        swe_ltm = np.full((WY_END - WY_START + 1, 365), np.nan)
        t_ltm = np.full((WY_END - WY_START + 1, 365), np.nan)
        for i, wy in enumerate(range(WY_START, WY_END + 1)):
            d = wy_daily(df, wy)
            swe_cm = d['WTEQ'].values * 100.0
            swe_ltm[i], t_ltm[i] = swe_cm, d['TAVG'].values
            res = snow_season_metrics(swe_cm)
            if res is None:
                continue
            p, f, m, z = res
            t_year, p_year = d['TAVG'], d['PRCPSA'] * 100.0  # degC, cm
            zeta.append(z)
            ra.append(p / (m - f))
            rm.append(p / (z - m))
            twy.append(t_year.mean() if t_year.notna().sum() >= 365 - MAX_MISSING_DAYS else np.nan)
            pwy.append(p_year.sum() if p_year.notna().sum() >= 365 - MAX_MISSING_DAYS else np.nan)

        zeta, twy, pwy = np.array(zeta, float), np.array(twy), np.array(pwy)
        joint = np.isfinite(twy) & np.isfinite(pwy)
        if joint.sum() < MIN_JOINT_YEARS:
            continue

        # observed sensitivity: zeta ~ [1, T, P]
        X, y = np.column_stack([np.ones(joint.sum()), twy[joint], pwy[joint]]), zeta[joint]
        coef, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ coef
        dof = len(y) - 3
        se = np.sqrt(np.linalg.inv(X.T @ X)[1, 1] * (resid @ resid) / dof)
        ci95 = stats.t.ppf(0.975, dof) * se

        # long-term mean cycles -> T0, T1, Tm
        with np.errstate(all='ignore'):
            s_bar, t_bar = np.nanmean(swe_ltm, axis=0), np.nanmean(t_ltm, axis=0)
        if np.isnan(t_bar).sum() > MAX_MISSING_DAYS or np.isnan(s_bar).sum() > MAX_MISSING_DAYS:
            continue
        T0, T1, t_rmse = fit_sine(doy, t_bar)
        s_sm = smooth11(s_bar)
        ds_ = np.diff(s_sm)
        t_mid = t_bar[:-1] + np.diff(t_bar) / 2  # midpoint T, as in E&E21's Fig2.m
        melting = (ds_ < 0) & (np.arange(364) >= np.nanargmax(s_sm))
        tm_station = np.nanmin(t_mid[melting]) if melting.any() else np.nan

        records.append(dict(
            code=code_, name=west.loc[code_, 'name'],
            lon=west.loc[code_].geometry.x, lat=west.loc[code_].geometry.y,
            elevation_m=west.loc[code_, 'elevation_m'], n_years=int(joint.sum()),
            dzdT_obs=coef[1], dzdT_ci95=ci95, T0=T0, T1=T1, T_sine_rmse=t_rmse,
            Ram=float(np.nanmean(np.array(ra) / np.array(rm))), Tm=tm_station,
            zeta_mean=float(np.nanmean(zeta)),
        ))
    station_df = pd.DataFrame(records).set_index('code')
    STATION_METRICS_CSV.parent.mkdir(exist_ok=True)
    station_df.to_csv(STATION_METRICS_CSV)
    print(f'Computed metrics for {len(station_df)} stations -> {STATION_METRICS_CSV}')

In [ ]:
# idealized-model sensitivity per station (E&E21 Fig2.m: per-station Ra/Rm, station-averaged Tm)
tm_bar = station_df['Tm'].mean()
station_df['dzdT_eq2'] = dzeta_dT0(station_df['T0'], station_df['T1'],
                                   ram=station_df['Ram'].values, tm=tm_bar)

print(f'stations passing QC:  {len(station_df)}   (E&E21 measured-T/P variant: 363)')
print(f'Ra/Rm  = {station_df.Ram.mean():.2f} +/- {station_df.Ram.std():.2f}    (E&E21: 0.34 +/- 0.14)')
print(f'Tm     = {tm_bar:.2f} +/- {station_df.Tm.std():.2f}    (E&E21: 0.18 +/- 2.02 degC)')
print(f'sine-fit RMSE = {station_df.T_sine_rmse.mean():.1f} degC      (E&E21: 1.4 degC)')
print(f'mean observed dzeta/dT0 = {station_df.dzdT_obs.mean():.1f} d/degC  (E&E21: -10.5 d/degC)')

## 5. E&E21 Fig. 2 — maps of observed vs idealized sensitivity

Both panels drop stations where either estimate is undefined, and plot the largest-magnitude values on top, as in E&E21. E&E21's pattern to look for: strongest sensitivity (deep blue/purple) in the Pacific Northwest, California, and the far southwest; weakest (yellow) in the continental interior (Colorado Rockies, Utah, Montana).

In [ ]:
plot_df = station_df.dropna(subset=['dzdT_obs', 'dzdT_eq2'])
proj = ccrs.LambertConformal(central_longitude=-114, central_latitude=40)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), subplot_kw={'projection': proj})
for ax, col, title, label in [(axes[0], 'dzdT_obs', 'Observations', 'a.'),
                              (axes[1], 'dzdT_eq2', 'Idealized model (Eq. 2)', 'b.')]:
    ax.set_extent([-125, -103, 31.5, 49.5], ccrs.PlateCarree())
    ax.add_feature(cfeature.STATES.with_scale('50m'), lw=0.5, edgecolor='0.3')
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'), lw=0.7)
    order = plot_df[col].sort_values(ascending=False).index  # most negative drawn last (on top)
    sc = ax.scatter(plot_df.loc[order, 'lon'], plot_df.loc[order, 'lat'],
                    c=plot_df.loc[order, col], s=16, vmin=-30, vmax=0, cmap='viridis',
                    transform=ccrs.PlateCarree(), zorder=3)
    ax.set_title(f'{label} {title}')
cb = fig.colorbar(sc, ax=axes, shrink=0.75, pad=0.02)
cb.set_label(r'$\partial\zeta/\partial T_0$ (days $^\circ$C$^{-1}$)')
fig.suptitle('Recreation of Evan & Eisenman (2021) Fig. 2 — SNOTEL-measured T/P, WY 2001-2018', y=0.98)
fig.savefig(FIG_DIR / 'ee2021_snotel_sensitivity_maps.png', dpi=250, bbox_inches='tight')
plt.show()

## 6. E&E21 Fig. 3a — station-by-station comparison

Observed regression sensitivity vs Eq. 2, with the 95% CI of the regression coefficient as error bars (their Fig. 3a). The relevant benchmark is E&E21's measured-T/P variant (their Supplementary Fig. 2a): $r=0.64$, bias 1.1, RMSE 5.8 days $^\circ$C$^{-1}$ over 363 stations. Their NARR-based main result was $r=0.77$ over 37 years; a weaker correlation is expected here from the shorter 18-year regressions. Fig. 3b (the VIC hydrologic-model check) is not recreated.

In [ ]:
ok = station_df.dropna(subset=['dzdT_obs', 'dzdT_eq2'])
r_all = np.corrcoef(ok.dzdT_obs, ok.dzdT_eq2)[0, 1]
inwin = ok[(ok.dzdT_eq2 > -40) & (ok.dzdT_obs > -40) & (ok.dzdT_obs < 0)]  # E&E21's Fig. 3a axis window
r_win = np.corrcoef(inwin.dzdT_obs, inwin.dzdT_eq2)[0, 1]
slope_win = np.polyfit(inwin.dzdT_eq2, inwin.dzdT_obs, 1)[0]
bias = (ok.dzdT_eq2 - ok.dzdT_obs).mean()
rmse = np.sqrt(((ok.dzdT_eq2 - ok.dzdT_obs) ** 2).mean())

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.errorbar(ok.dzdT_eq2, ok.dzdT_obs, yerr=ok.dzdT_ci95, fmt='o', ms=4,
            color='tab:blue', ecolor='lightsteelblue', elinewidth=1, zorder=3)
ax.plot([-40, 0], [-40, 0], 'k--', lw=1)
ax.set_xlim(-40, 0); ax.set_ylim(-40, 0)
ax.grid(alpha=0.3)
ax.set_xlabel(r'$\partial\zeta/\partial T_0$ from Eq. 2 (days $^\circ$C$^{-1}$)')
ax.set_ylabel(r'$\partial\zeta/\partial T_0$ observed (days $^\circ$C$^{-1}$)')
ax.set_title('Recreation of Evan & Eisenman (2021) Fig. 3a')
ax.annotate(f'n = {len(ok)}\nr = {r_all:.2f} (all) / {r_win:.2f} (plot window)\n'
            f'slope = {slope_win:.2f}\nbias = {bias:.1f}, RMSE = {rmse:.1f} d $^\\circ$C$^{{-1}}$',
            xy=(0.03, 0.03), xycoords='axes fraction', fontsize=9,
            bbox=dict(boxstyle='round', fc='w', alpha=0.8))
fig.savefig(FIG_DIR / 'ee2021_observed_vs_idealized_sensitivity.png', dpi=250, bbox_inches='tight')
plt.show()
print(f'r={r_all:.2f} all / r={r_win:.2f} in plot window; slope={slope_win:.2f}; '
      f'bias={bias:.1f}; RMSE={rmse:.1f}  (E&E21 measured variant: r=0.64, bias=1.1, RMSE=5.8, slope=1.0+/-0.1)')

## 7. Global climatological $T_0$ and $T_1$ (for E&E21 Fig. 4)

E&E21 computed $T_0$ and $T_1$ from the 1982-2018 monthly-mean MERRA-2 temperature at 0.5$^\circ$. Here we fit Eq. 1 to the ERA5 1990-2019 day-of-year 2 m temperature climatology from [WeatherBench2](https://weatherbench2.readthedocs.io/en/latest/data-guide.html) at $\sim$0.7$^\circ$ (equiangular, conservatively regridded) — freely readable over anonymous HTTPS, consistent with the ERA5 data used elsewhere in this repo. The $\sim$770 MB read happens once; the fitted $T_0$/$T_1$ fields are cached as a small netCDF. The land mask (E&E21 masks land fraction < 0.1) is rasterized from Natural Earth 110m land polygons.

In [ ]:
WB2_CLIM_URL = ('https://storage.googleapis.com/weatherbench2/datasets/era5-hourly-climatology/'
                '1990-2019_6h_512x256_equiangular_conservative.zarr')

if ERA5_CLIM_NC.exists():
    clim = xr.open_dataset(ERA5_CLIM_NC)
    print(f'Loaded cached T0/T1 climatology from {ERA5_CLIM_NC}')
else:
    print('Streaming ERA5 day-of-year climatology from WeatherBench2 (~770 MB, one time)...')
    t2m = xr.open_zarr(WB2_CLIM_URL)['2m_temperature'].mean('hour') - 273.15  # daily-mean climatology [degC]
    t2m = t2m.compute()
    t2m = t2m.assign_coords(longitude=np.where(t2m.longitude > 180, t2m.longitude - 360, t2m.longitude))
    t2m = t2m.sortby('longitude').transpose('dayofyear', 'latitude', 'longitude')

    d = t2m.dayofyear.values.astype(float)
    X = np.column_stack([np.ones_like(d), np.sin(OMEGA * d), np.cos(OMEGA * d)])
    coef, *_ = np.linalg.lstsq(X, t2m.values.reshape(len(d), -1), rcond=None)
    ny, nx = t2m.sizes['latitude'], t2m.sizes['longitude']
    clim = xr.Dataset(
        {'T0': (('latitude', 'longitude'), coef[0].reshape(ny, nx).astype('float32')),
         'T1': (('latitude', 'longitude'), np.hypot(coef[1], coef[2]).reshape(ny, nx).astype('float32'))},
        coords={'latitude': t2m.latitude, 'longitude': t2m.longitude},
        attrs={'source': WB2_CLIM_URL,
               'description': ('Sinusoid fit T(d) = T0 + a sin(wd) + b cos(wd) to the ERA5 1990-2019 '
                               'day-of-year 2m temperature climatology; T1 = hypot(a, b). '
                               'Recreation of Evan & Eisenman (2021) Fig. 4 inputs (they used MERRA-2 1982-2018).')})
    ERA5_CLIM_NC.parent.mkdir(exist_ok=True)
    clim.to_netcdf(ERA5_CLIM_NC)
    print(f'Saved -> {ERA5_CLIM_NC}')

# land mask from Natural Earth 110m land polygons (E&E21: mask land fraction < 0.1)
lon, lat = clim.longitude.values, clim.latitude.values
dx, dy = np.diff(lon).mean(), np.diff(lat).mean()
transform = rasterio.transform.from_origin(lon.min() - dx / 2, lat.max() + dy / 2, dx, dy)
land_geoms = shapereader.Reader(
    shapereader.natural_earth(resolution='110m', category='physical', name='land')).geometries()
land = rasterio.features.rasterize(((g, 1) for g in land_geoms), out_shape=(len(lat), len(lon)),
                                   transform=transform, all_touched=True).astype(bool)[::-1]  # -> ascending lat
clim['land'] = (('latitude', 'longitude'), land)

clim['dzdT0'] = xr.DataArray(dzeta_dT0(clim.T0.values, clim.T1.values),
                             coords=clim.T0.coords).where(clim.land)
nh = clim.dzdT0.sel(latitude=slice(25, 85))
print(f'NH 25-85N sensitivity percentiles (5/50/95): '
      f'{np.nanpercentile(nh.values, [5, 50, 95]).round(1)} days/degC')

## 8. E&E21 Fig. 4 — global sensitivity of snow disappearance timing

Eq. 2 applied globally with the western-US station values $R_a/R_m = 0.34$ and $T_m = 0.18\,^\circ$C, exactly as in E&E21. White land = undefined ($T_1 < |T_0 - T_m|$: temperature never crosses the melt threshold — e.g. the tropics, Sahara, Antarctic interior). E&E21's Fig4.m stretches the colormap nonlinearly to add contrast at the weak end; we replicate that with viridis. Compare against: largest magnitudes on coasts (Pacific NW, western Europe, Greenland margin), the Arctic (>75$^\circ$N), and near the southern edge of the defined region; $-3$ to $-6$ days $^\circ$C$^{-1}$ in the interiors of North America and Eurasia; below $-30$ days $^\circ$C$^{-1}$ in southern South America.

In [ ]:
def stretched_cmap(name='viridis', n=256):
    """Nonlinear colormap stretch used in E&E21's Fig4.m (cumulative-sum index warping)."""
    w = np.cumsum(np.arange(1, n + 1)).astype(float)
    return matplotlib.colors.ListedColormap(plt.get_cmap(name)(w / w[-1]))

cmap4 = stretched_cmap()

fig = plt.figure(figsize=(13, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1.25, 1], hspace=0.25, wspace=0.15)

# (a) Northern Hemisphere map
ax_a = fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree())
ax_a.set_extent([-180, 180, 25, 85], ccrs.PlateCarree())
pm = ax_a.pcolormesh(clim.longitude, clim.latitude, clim.dzdT0, vmin=-21, vmax=-3,
                     cmap=cmap4, transform=ccrs.PlateCarree())
ax_a.coastlines(lw=0.6)
gl = ax_a.gridlines(draw_labels=['left', 'bottom'], lw=0.3, color='0.7')
cb = fig.colorbar(pm, ax=ax_a, shrink=0.85, pad=0.02, ticks=np.arange(-21, -2, 3))
cb.set_label(r'$\partial\zeta/\partial T_0$ (days $^\circ$C$^{-1}$)')
ax_a.set_title(r'a. Global $\partial\zeta/\partial T_0$ from Eq. 2 '
               rf'($R_a/R_m$={RAM_GLOBAL}, $T_m$={TM_GLOBAL}$^\circ$C)')

# (b) zonal-mean transect, Northern Hemisphere
ax_b = fig.add_subplot(gs[1, 0])
with np.errstate(all='ignore'):
    zonal = clim.dzdT0.sel(latitude=slice(0.1, 90)).mean('longitude', skipna=True)
ax_b.plot(zonal, zonal.latitude, lw=2)
ax_b.set_xlim(-50, 0); ax_b.set_ylim(20, 90); ax_b.set_yticks(np.arange(30, 91, 15))
ax_b.grid(alpha=0.3)
ax_b.set_xlabel(r'$\partial\zeta/\partial T_0$ (days $^\circ$C$^{-1}$)')
ax_b.set_ylabel(r'Latitude ($^\circ$N)')
ax_b.set_title('b. Zonal mean')

# (c) southern South America
ax_c = fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree())
ax_c.set_extent([-80, -60, -60, -20], ccrs.PlateCarree())
pm_c = ax_c.pcolormesh(clim.longitude, clim.latitude, clim.dzdT0, vmin=-40, vmax=-10,
                       cmap=cmap4, transform=ccrs.PlateCarree())
ax_c.coastlines(lw=0.6)
ax_c.gridlines(draw_labels=['left', 'bottom'], lw=0.3, color='0.7')
cb_c = fig.colorbar(pm_c, ax=ax_c, shrink=0.9, pad=0.02)
cb_c.set_label(r'$\partial\zeta/\partial T_0$ (days $^\circ$C$^{-1}$)')
ax_c.set_title('c. Southern South America')

fig.suptitle('Recreation of Evan & Eisenman (2021) Fig. 4 (ERA5 climatology in place of MERRA-2)', y=0.96)
fig.savefig(FIG_DIR / 'ee2021_global_sensitivity.png', dpi=250, bbox_inches='tight')
plt.show()

## 9. Comparison with this repo's spring temperature sensitivity

The two "days per $^\circ$C" numbers measure related but distinct things:

| | Evan & Eisenman (2021) | This repo |
|---|---|---|
| response variable | snow **disappearance** date $\zeta$ (SWE $\rightarrow$ 0) | snowmelt **runoff onset** (Sentinel-1, roughly peak-melt timing) |
| temperature variable | **annual-mean** $T_0$ (climatological) | **spring** $T$ anomaly (ERA5-Land, per water year) |
| estimate | idealized model / multi-decade regression | Theil-Sen slope over WY 2015-2024 anomalies |

Runoff onset precedes full disappearance, and spring $T$ varies more than annual-mean $T$, so the magnitudes need not match 1:1 — but if E&E21's mechanism (distance of the annual cycle from the melt threshold) also shapes runoff-onset sensitivity, the *spatial patterns* should correlate: mountain ranges the idealized model flags as sensitive should also show stronger observed runoff-onset responses. Here we average the recreated global $\partial\zeta/\partial T_0$ field over each GMBA mountain range and compare with the observed per-range sensitivity (`anomaly_slope`) from `mountain_range_era5_analysis.ipynb`.

In [ ]:
try:
    gmba = gsro_stats.range_metrics_gdf(config.version)   # GMBA polygons + results/<version>/mountain_range_metrics.csv
except FileNotFoundError as e:
    gmba = None
    print(f'{e} - skipping comparison.')
if gmba is not None:
    gmba = gmba[gmba['anomaly_slope'].notna() & (gmba['anomaly_n'] >= 8)].reset_index(drop=True)

    dz_vals = clim.dzdT0.values
    rows = []
    for _, row in gmba.iterrows():
        mask = rasterio.features.rasterize([(row.geometry, 1)], out_shape=dz_vals.shape,
                                           transform=transform, all_touched=True).astype(bool)[::-1]
        vals = dz_vals[mask]
        vals = vals[np.isfinite(vals)]
        if len(vals) >= 3:
            rows.append((row['MapName'], vals.mean(), row['anomaly_slope'], row['anomaly_corr']))
    comp = pd.DataFrame(rows, columns=['MapName', 'ee2021_dzdT0', 'runoff_onset_sensitivity',
                                       'runoff_onset_corr']).set_index('MapName')

    r_p = comp.ee2021_dzdT0.corr(comp.runoff_onset_sensitivity)
    r_s = comp.ee2021_dzdT0.corr(comp.runoff_onset_sensitivity, method='spearman')

    fig, ax = plt.subplots(figsize=(7.5, 7))
    ax.scatter(comp.ee2021_dzdT0, comp.runoff_onset_sensitivity, s=30, alpha=0.8)
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), 0]
    ax.plot(lims, lims, 'k--', lw=1, label='1:1')
    ax.axhline(0, color='0.7', lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_xlabel(r'Evan & Eisenman (2021) idealized $\partial\zeta/\partial T_0$ (days $^\circ$C$^{-1}$)')
    ax.set_ylabel(r'Observed runoff-onset sensitivity to spring T anomaly (days $^\circ$C$^{-1}$)')
    ax.set_title(f'Mountain-range comparison (n={len(comp)}): r={r_p:.2f}, Spearman r={r_s:.2f}')
    for name in comp.ee2021_dzdT0.nsmallest(3).index.union(comp.runoff_onset_sensitivity.nsmallest(3).index):
        ax.annotate(name, (comp.loc[name, 'ee2021_dzdT0'], comp.loc[name, 'runoff_onset_sensitivity']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.legend()
    fig.savefig(FIG_DIR / 'runoff_onset_sensitivity_vs_ee2021.png', dpi=250, bbox_inches='tight')
    plt.show()

    print(comp.sort_values('ee2021_dzdT0').round(1).head(15))

## Takeaways

**Recreation fidelity** (all benchmark numbers printed by the cells above):

| Quantity | Evan & Eisenman (2021) | This recreation |
|---|---|---|
| E&E21 Fig. 1: $\zeta$, $R_a$, $R_m$ (Virginia Lakes Ridge, WY2014/2017) | 226/260, 0.17/0.69, 0.56/1.80 | identical — same underlying NRCS data |
| station-mean $R_a/R_m$ | 0.34 $\pm$ 0.14 | 0.32 $\pm$ 0.09 |
| station-mean $T_m$ | 0.18 $\pm$ 2.02 $^\circ$C | 0.33 $\pm$ 1.25 $^\circ$C |
| sinusoid-fit RMSE of daily $T$ | 1.4 $^\circ$C | 1.6 $^\circ$C |
| mean observed $\partial\zeta/\partial T_0$ | $-10.5$ d $^\circ$C$^{-1}$ | $-9.1$ d $^\circ$C$^{-1}$ |
| obs vs Eq. 2 correlation | 0.77 (NARR, 37 yr); **0.64 (measured $T$/$P$, 18 yr)** | 0.31 all stations / 0.58 within E&E21's Fig. 3a axis window |
| obs vs Eq. 2 slope / bias / RMSE | 1.0 $\pm$ 0.1 / 1.1 / 5.8 | 1.10 / $-0.3$ / 8.3 |
| E&E21 Fig. 4 global structure | most sensitive: coasts, Arctic, western US, Central Europe, southern South America ($< -30$); interiors $-3$ to $-6$ | same structure from ERA5 |

The bolded row is the appropriate benchmark: like E&E21's Supplementary Fig. 2a variant we regress on station-measured $T$/$P$ over 18 water years, so the per-station regression is inherently noisy (median 95% CI $\approx \pm 7$ d $^\circ$C$^{-1}$ — the error bars in Fig. 3a). The all-station $r$ is dragged down by a few stations where $T_1 \approx |T_0 - T_m|$ sends Eq. 2 toward its singularity; inside E&E21's own plot window the agreement (r = 0.58, slope = 1.10) is close to their published measured-data result.

**Comparison with this repo's spring temperature sensitivity.** Across 101 GMBA mountain ranges, the recreated idealized $\partial\zeta/\partial T_0$ correlates with the observed Sentinel-1 runoff-onset sensitivity to spring temperature anomalies at r = 0.47 (Spearman 0.51). Observed runoff-onset sensitivities are systematically smaller in magnitude than the idealized disappearance-date sensitivities (points sit above the 1:1 line) — expected, since runoff onset occurs near peak melt rather than at snow disappearance, and a 10-year anomaly regression is a different estimator from a climatological derivative. But the ranges the mechanism flags as most vulnerable (Klamath Mountains, the BC Insular Mountains, Patagonian Andes, Mediterranean/Balkan ranges) are indeed among the more responsive in the satellite record — supporting E&E21's core claim that proximity of the annual temperature cycle to the melt threshold, not just mean warmth, controls where snowmelt timing responds fastest to warming.

**Outputs**

* figures: `figures/temp_sensitivity/evan_eisenman_2021/`
* per-station metrics: `csvs/evan_eisenman_2021_snotel_station_metrics.csv` (delete to recompute)
* global $T_0$/$T_1$ climatology: `era5_data/evan_eisenman_2021_era5_T0_T1_climatology.nc` (delete to re-stream from WeatherBench2)
* SNOTEL daily archive cache: `data/snotel_ccss_archive/` (delete to re-download)